# 01 · LLM 基础

> RAG 建立在大语言模型之上。先把 LLM 的几个底层概念讲清，后面讲 RAG 才有抓手。

**本文件覆盖知识点**：Transformer / Attention(多头) / Encoder-Decoder / Token与Tokenizer / Context Window / 位置编码(RoPE) / 采样参数(Temperature·Top-P) / Prompt / Function Calling / 结构化输出 / 幻觉 / 长上下文

> 篇幅所限，这里用「一句话原理 + 可运行小实验」的方式带过，目的不是复现训练，而是建立正确的直觉。

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 从 Token 说起

LLM 不直接读字，而是先把文本切成**最小处理单元（Token）**：
- 英文常按词/子词切：`hello` → `hello`；`running` → `run` + `ning`；
- 中文常按字/词切：`检索` 可能切成 1~3 个 token（不同分词器不同）；
- **Tokenizer（分词器）**负责这件「文本 ↔ token id」的转换。

为什么要懂 token？因为它决定：
- **Context Window（上下文窗口）**按 token 计——一次能放进去多少字，取决于 tokenizer 怎么切；
- 计费按 token 算，RAG 里塞多少上下文直接决定成本。

In [1]:
# 调用“百炼的分词接口”：看同一句话被切成多少个 token（窗口/成本都按 token 计）
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
if not API_KEY or '你的' in API_KEY:
    print('⚠ 请先在项目根目录 .env 填入 DASHSCOPE_API_KEY（复制 .env.example 为 .env 即可）')
    API_KEY = None

text = '检索增强生成（RAG）把大模型与外置知识库结合'
print('原文:', text)
print('人类视角字符数:', len(text))

if API_KEY:
    from dashscope import Tokenization
    # 百炼 Tokenization 接口：返回这句文本的 tokens 列表 + token_ids
    resp = Tokenization.call(model='qwen-plus', prompt=text, api_key=API_KEY)
    if resp.status_code == 200:
        out = resp.output if isinstance(resp.output, dict) else vars(resp.output)
        tokens = out.get('tokens')
        token_ids = out.get('token_ids')
        print('分词结果 tokens:', tokens)
        n = len(token_ids) if token_ids else (len(tokens) if tokens else None)
        if n is not None:
            print(f'→ 这句话被切成 {n} 个 token')
            print('含义: 上下文窗口按 token 计、成本按 token 计——RAG 要做的正是“少而准”地塞 token。')
        usage = getattr(resp, 'usage', None)
        if usage:
            print('usage:', usage)
    else:
        print('调用失败:', getattr(resp, 'code', ''), getattr(resp, 'message', ''))
else:
    print('（未配置 Key，已跳过在线分词；填入 Key 后即可看到模型的真实切分。）')

原文: 检索增强生成（RAG）把大模型与外置知识库结合
人类视角字符数: 23
分词结果 tokens: ['检索', '增强', '生成', '（', 'R', 'AG', '）', '把', '大', '模型', '与', '外', '置', '知识', '库', '结合']
→ 这句话被切成 16 个 token
含义: 上下文窗口按 token 计、成本按 token 计——RAG 要做的正是“少而准”地塞 token。
usage: {'input_tokens': 16}


## 2. Transformer 与 Attention

- **Transformer** 是当代 LLM 的统一底座：一种基于「注意力」的神经网络结构；
- **Self-Attention（自注意力）**让每个词在编码时能看到句子里其它所有词，并给它们分配不同权重——
  例如「它」应该更多关注「星云客服机器人」而不是「部署」；
- **Multi-Head Attention（多头注意力）**并行地用多组权重去关注不同维度的关系（语法关系、指代关系、语义关系…），再合并；
- **位置编码**：注意力本身不分先后，所以必须给每个 token 注入位置信息。现代模型多用 **RoPE（旋转位置编码）**，它天然支持长序列外推。

> 为什么 RAG 要懂它？因为「把哪些片段拼在上下文里、拼在什么顺序」本质上是在喂给注意力机制——
> 所以上下文里噪声越多、相关片段被埋得越深，模型越容易答错（这正是后面“上下文工程”要解决的）。

## 3. 采样参数：Temperature / Top-P

模型每一步都在为「下一个 token」生成一个概率分布，然后**采样**决定实际输出：
- **Temperature**：控制分布的“锐利程度”。温度低 → 更保守、更确定；温度高 → 更大胆、更多样。
- **Top-P（核采样）**：只在累计概率达到 P 的那一小撮候选里采样，去掉长尾。

> RAG 场景一般希望“忠于资料、稳定作答”，所以常把温度调低（如 0.1~0.3）。

In [4]:
import numpy as np

def temperature_softmax(logits, temperature):
    """把 logits 变成概率分布，temperature 越高分布越平坦"""
    logits = np.array(logits) / max(temperature, 1e-5)
    e = np.exp(logits - logits.max())  # 减最大值防溢出
    return e / e.sum()

logits = [0.2, 0.9, 2.1, 1.3, 0.1]  # 假设模型对 5 个候选词的打分
for t in (0.2, 1.0, 2.0):
    p = temperature_softmax(logits, t)
    print(f'temperature={t}: 概率分布={np.round(p, 3)}  | 最可能词 index={int(np.argmax(p))}')
print('\n温度越低越“一锤定音”，越高越“天马行空”。')

temperature=0.2: 概率分布=[0.    0.002 0.98  0.018 0.   ]  | 最可能词 index=2
temperature=1.0: 概率分布=[0.073 0.148 0.491 0.221 0.066]  | 最可能词 index=2
temperature=2.0: 概率分布=[0.13  0.185 0.336 0.225 0.124]  | 最可能词 index=2

温度越低越“一锤定音”，越高越“天马行空”。


In [ ]:
# 知识点·真调说明：Temperature —— 让模型自己示范「低温 vs 高温」的输出差别
# （上面 numpy 是“数学上的采样”，这里看真实模型在同一道题上的表现）
print('① temperature=0.1（低 · 更确定、更收敛）')
_llm_live(
    prompt='给一个做“宠物领养咨询”的 RAG 客服机器人起 3 个中文名字，每个名字用一句话解释。',
    system='你是给 RAG 客服机器人起名的助手，答案务实、稳妥、不追求新奇。',
    fallback='未配置 Key 的固定样例：\n'
             '- 领养小助手：一句话解答领养问题\n'
             '- 宠物问诊台：快速回答宠物健康问题\n'
             '- 领养百晓生：覆盖领养全流程问答',
    temperature=0.1,
)
print()
print('② temperature=1.5（高 · 更大胆、更多样，也可能跑题/造词）')
_llm_live(
    prompt='给一个做“宠物领养咨询”的 RAG 客服机器人起 3 个中文名字，每个名字用一句话解释。',
    system='你是给 RAG 客服机器人起名的创意助手，答案放开想象力，允许夸张、有趣的名字。',
    fallback='未配置 Key 的固定样例：\n'
             '- 毛球救火队：领养问题秒级出警\n'
             '- 铲屎官外挂：从选猫到带回家的全程外挂\n'
             '- 汪汪喵喵雷达：扫一眼就知道该领养谁',
    temperature=1.5,
)
print()
print('对比同一条 prompt：低温稳定可控，高温更发散、更大胆。')
print('→ RAG 知识问答要“忠于资料、稳定作答”，所以生成时把 temperature 调低（0.1~0.3）。')

## 4. Prompt / Function Calling / 结构化输出

- **Prompt（提示词）**：你发给模型的指令文本。RAG 的“检索→拼上下文→回答”本质就是一套精心设计的 prompt；
- **Function Calling / Tool Calling（工具调用）**：模型输出「要调用的函数名+参数」的 JSON，由外部系统真正执行——这是 Agentic RAG（见 28 课）能“自己决定检索/搜索”的基础；
- **Structured Output（结构化输出）**：要求模型输出符合 schema 的 JSON，是让 RAG 下游（改写、重排、评测打分）可靠对接的关键。

## 5. 幻觉与长上下文

- **幻觉（Hallucination）**：模型对不知道的内容“一本正经地编造”——RAG 想缓解的核心问题之一（第 26 课深入）；
- **Long Context（长上下文）**：新一代模型支持百万级上下文，但“塞得下”不等于“用得好”（位置靠中间的信息容易丢，见第 39 课 Lost-in-the-Middle）。



In [ ]:
# 知识点·真调说明：Prompt 设计 —— 同一问题，prompt 越讲究，答案越可用
print('① 随意版 prompt（只说“是什么 / 有什么用”）')
_llm_live(
    prompt='RAG 是什么？有什么用？',
    system='直接回答用户问题。',
    fallback='未配置 Key 的固定样例：\nRAG 是检索增强生成（Retrieval-Augmented Generation），'
             '在生成回答前先从外部知识库检索相关内容并拼进上下文，能减少幻觉、补充私有知识。',
    temperature=0.2,
)
print()
print('② 讲究版 prompt（给定角色 + 约束 + 结构 + 例子要求）')
_llm_live(
    prompt='请向一位产品经理解释“RAG 是什么、解决什么问题、相比直接问大模型好在哪”。',
    system='你是 RAG 技术讲师。回答要求：不超过 4 句话；用 ①②③ 列点；必须给出一个具体业务例子。',
    fallback='未配置 Key 的固定样例：\n'
             '① RAG=检索增强生成：回答前先检索企业知识库，把相关资料拼进上下文再作答。\n'
             '② 解决的问题：大模型没学过企业私有/新知识，会“一本正经胡说”（幻觉）。\n'
             '③ 相比直接问：答案有资料出处、可溯源、更贴近真实业务。\n'
             '例：客服机器人先查产品手册再作答，而不是凭空猜退换货政策。',
    temperature=0.2,
)
print('同一问题，prompt 决定回答质量——RAG 的 system prompt / 上下文拼装同样是“精心的 prompt”。')

In [ ]:
# 知识点·真调说明：结构化输出 —— 让模型只返回能 json.loads 直接解析的 JSON
import json as _json
out = _llm_live(
    prompt='请把下面这句话提炼成一个 JSON 对象：\n'
           '“检索增强生成（RAG）让模型在回答前先检索企业知识库，从而减少幻觉并给出可溯源的答案。”',
    system='只输出一个 JSON 对象，禁止输出任何其它文字、解释或代码块标记。字段结构：'
           '{"title": 字符串标题, "summary": 一句话摘要, "tags": 字符串标签数组}',
    fallback='未配置 Key 的固定样例：\n'
             '{"title": "什么是检索增强生成", '
             '"summary": "RAG 让模型回答前先检索企业知识库，减少幻觉并提供可溯源答案。", '
             '"tags": ["RAG", "检索", "防幻觉"]}',
    temperature=0.1,
)
if out is None:
    out = ('{"title": "什么是检索增强生成", '
           '"summary": "RAG 让模型回答前先检索企业知识库，减少幻觉并提供可溯源答案。", '
           '"tags": ["RAG", "检索", "防幻觉"]}')
    print('（以上为固定样例；下面用样例演示 json.loads 校验）')
try:
    obj = _json.loads(out)
    print('json.loads 校验通过 ✅ 顶层字段：', list(obj))
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明约束不够严，需在 prompt 里再收紧。')
print('→ 结构化输出让“模型结果”能被程序直接接管，是 RAG 下游（改写 / 重排 / 评测打分）可靠对接的基础。')

In [ ]:
# 知识点·真调说明：Function Calling —— 模型“不执行，只表态要调哪个函数”，由系统去真查
_llm_live(
    prompt='用户问：“你们产品支持导出 Excel 吗？”——你并没有在训练语料里见过“星云”这份产品手册，'
           '要回答就必须先去查知识库。请输出你的下一步动作。',
    system='你是一个 RAG 系统的“意图路由器”。规则：'
           '如果需要查知识库，只输出一个 JSON：{"tool": "kb_search", "query": "<改写后的检索词>"}；'
           '如果不需要查库就能答，只输出：{"tool": null, "answer": "<直接回答>"}。禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n{"tool": "kb_search", "query": "星云 产品 导出 Excel 是否支持"}',
    temperature=0.1,
)
print('模型交出的不是答案，而是“该调 kb_search、用什么 query”的 JSON——由外部系统真正执行检索。')
print('→ 这就是 Function Calling / Tool Calling 的最小雏形；28 课 Agentic RAG 会把“调用-执行-回填”跑完整。')

In [ ]:
# 知识点·真调说明：幻觉 —— 问模型一个它“不可能知道”的私有细节，观察是否一本正经编造
_llm_live(
    prompt='用户张伟上周四（2026-09-03）通过“星云”客户服务系统提交的工单标题是什么？工单号是多少？',
    system='你是星云客服。请如实回答；你不知道的事情就明确说“无法确认”，绝不要编造。',
    fallback='未配置 Key 的固定样例（观察重点）：\n'
             '若模型回答“工单标题为‘登录页面报错’，工单号 SW-2041。”——细节越具体越可疑：'
             '模型没有访问工单系统的通道，这段多半是“一本正经的编造”（幻觉）。\n'
             '若模型回答“抱歉，我无法访问工单系统，无法确认。”——这是守住了边界。',
    temperature=0.1,
)
print('→ 模型对“私有、未见过、无法核验”的事实没有可靠答案来源；RAG 就是先检索出依据再作答，'
      '从源头压住幻觉（26 课深入）。')

## 小结

LLM 的每个特性都在约束 RAG 的设计：token 决定成本与容量、注意力决定上下文质量、采样决定稳定性、Function Calling 开启 Agentic。下一课回答“为什么一定要 RAG”。